In [2]:
import csv
import json
import sys
from datetime import datetime

import requests


# =========================
# Configuration
# =========================

BASE_URL = "https://api.tickleright.in/go/api/v1/open/key/inquiry-demo-followup"
BEARER_TOKEN = "TR1TIWMWD28Oy7siniSXp1wWdbWDfr4CXNrietYextrpk2oqGOiR5igOjAV4AHgpx2uQ0fDpULnyMtY0mNkcU1dONNAXGx6czTA8TF89JsOAPKNZS2Ws7UoTo7CRv4EPDkw0JGLXXDC7AQshLmO50DcOf4o0G8v8oZ8l75GtWLUM0uP8e5tgwZJLnACcEyBjIkL8PFIOIuMJUmT3TV0rXJI8WvJ3UlT6IV8ULMRgnHcFrN6n2HK8mFWIRkO88Qws6J"

FILE_NAME = "inquiries_data"

# True  = save records in a CSV file
# False = print records in the console
CSV_DOWNLOAD = True

# None = fetch all available records
# Or specify a number, such as 25000
MAX_RECORDS = None

# Maximum records requested in each API call
PAGE_LIMIT = 10_000


def format_value(value):
    """Convert API values into CSV-safe values."""
    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)

    return value


def get_all_data():
    request_headers = {
        "Authorization": f"Bearer {BEARER_TOKEN}"
    }

    offset = 0
    total_records = 0
    column_names = None

    output_file = None
    csv_file = None
    writer = None

    try:
        if CSV_DOWNLOAD:
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            output_file = f"{FILE_NAME}_{timestamp}.csv"

        while True:
            if MAX_RECORDS is not None:
                remaining_records = MAX_RECORDS - total_records

                if remaining_records <= 0:
                    print(
                        "Requested record limit reached.",
                        file=sys.stderr
                    )
                    break

                current_limit = min(PAGE_LIMIT, remaining_records)
            else:
                current_limit = PAGE_LIMIT

            params = {
                "limit": current_limit,
                "offset": offset
            }

            print(
                f"Fetching: offset={offset}, limit={current_limit}",
                file=sys.stderr
            )

            response = requests.get(
                BASE_URL,
                headers=request_headers,
                params=params,
                timeout=120
            )

            response.raise_for_status()
            data = response.json()

            if not isinstance(data, list):
                raise ValueError(
                    "Unexpected API response. Expected a list of records."
                )

            if not data:
                print("No more records available.", file=sys.stderr)
                break

            # Create the writer and write headings only once
            if column_names is None:
                column_names = list(data[0].keys())

                if CSV_DOWNLOAD:
                    csv_file = open(
                        output_file,
                        mode="w",
                        newline="",
                        encoding="utf-8-sig"
                    )
                    output_destination = csv_file
                else:
                    output_destination = sys.stdout

                writer = csv.DictWriter(
                    output_destination,
                    fieldnames=column_names,
                    extrasaction="ignore"
                )

                writer.writeheader()

            # Write or print the current batch
            for record in data:
                formatted_record = {
                    column: format_value(record.get(column))
                    for column in column_names
                }

                writer.writerow(formatted_record)

            if csv_file is not None:
                csv_file.flush()

            fetched_count = len(data)
            total_records += fetched_count
            offset += fetched_count

            print(
                f"Fetched {fetched_count} records. "
                f"Total: {total_records}",
                file=sys.stderr
            )

            if fetched_count < current_limit:
                print("Last page reached.", file=sys.stderr)
                break

        print(
            f"Finished. Total records: {total_records}",
            file=sys.stderr
        )

        if CSV_DOWNLOAD and output_file and total_records > 0:
            print(
                f"CSV file created: {output_file}",
                file=sys.stderr
            )

    except requests.RequestException as error:
        print(f"API request failed: {error}", file=sys.stderr)

        if getattr(error, "response", None) is not None:
            print(
                f"API response: {error.response.text}",
                file=sys.stderr
            )

    except (ValueError, json.JSONDecodeError) as error:
        print(f"Invalid API response: {error}", file=sys.stderr)

    except OSError as error:
        print(f"File-writing error: {error}", file=sys.stderr)

    finally:
        if csv_file is not None:
            csv_file.close()


if __name__ == "__main__":
    get_all_data()

Fetching: offset=0, limit=10000
Fetched 10000 records. Total: 10000
Fetching: offset=10000, limit=10000
Fetched 10000 records. Total: 20000
Fetching: offset=20000, limit=10000
Fetched 10000 records. Total: 30000
Fetching: offset=30000, limit=10000
Fetched 10000 records. Total: 40000
Fetching: offset=40000, limit=10000
Fetched 10000 records. Total: 50000
Fetching: offset=50000, limit=10000
Fetched 10000 records. Total: 60000
Fetching: offset=60000, limit=10000
Fetched 9582 records. Total: 69582
Last page reached.
Finished. Total records: 69582
CSV file created: inquiries_data_2026-09-01_16-33-13.csv
